In [1]:
def slice_from_ref_region(dset, ref_region_row):
    start = int(ref_region_row["start"])
    stop  = int(ref_region_row["stop"])
    return dset[start:stop]
def get_event_hits_by_event_index(f, event_index):
    hits = f["charge/calib_prompt_hits/data"]
    rr   = f["charge/events/ref/charge/calib_prompt_hits/ref_region"]
    return slice_from_ref_region(hits, rr[event_index])

In [18]:
import numpy as np
import h5py


file = "data/packet-0060070-2025_10_31_15_57_27_CDT.FLOW.hdf5"

pixel_seen = {}
pixel_abnormal = {}

MULTIPLICITY_THRESHOLD = 4

with h5py.File(file,"r") as f:
    events = f["charge/events/data"]
    hits = f["charge/calib_prompt_hits/data"]
    rr = f["charge/events/ref/charge/calib_prompt_hits/ref_region"]

    for evt_idx in range(len(events)):

      start = rr[evt_idx]["start"]
      stop  = rr[evt_idx]["stop"]

      hits_ev = hits[start:stop]

      if len(hits_ev) == 0:
          continue


      counts = {}

      for i in range(len(hits_ev)):
        p = (
        int(hits_ev["io_group"][i]),
        int(hits_ev["io_channel"][i]),
        int(hits_ev["chip_id"][i]),
        int(hits_ev["channel_id"][i])
        )
        counts[p] = counts.get(p, 0) + 1

      # update global stats
      for p, n in counts.items():

        # pixel_seen
        pixel_seen[p] = pixel_seen.get(p, 0) + 1

                # abnormal count
        if n > MULTIPLICITY_THRESHOLD:
          pixel_abnormal[p] = pixel_abnormal.get(p, 0) + 1

In [20]:
count_bad = 0

for p in pixel_seen:

    seen = pixel_seen[p]
    abnormal = pixel_abnormal.get(p, 0)

    if seen > 200 and abnormal > 100:
        print(
            "BAD PIXEL:",
            p,
            "| seen =", seen,
            "| abnormal =", abnormal
        )
        count_bad += 1

print("Total bad pixels =", count_bad)
        

BAD PIXEL: (2, 29, 28, 18) | seen = 19748 | abnormal = 16809
Total bad pixels = 1
